In [ ]:
from __future__ import annotations

from collections import deque
from dataclasses import dataclass
from typing import Dict, List, Tuple
import warnings

import numpy as np
import pandas as pd
import joblib


from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import GridSearchCV, train_test_split


def configure_runtime_warnings() -> None:
    warnings.filterwarnings("ignore", category=UserWarning, module=r"joblib\..*")
    warnings.filterwarnings("ignore", message=r".*resource_tracker.*", category=UserWarning)
    warnings.filterwarnings("ignore", category=FutureWarning, module=r"sklearn\..*")


configure_runtime_warnings()
print("Setup complete.")

Setup complete.


In [19]:
# Streaming slope buffers
N = 10                      # last 10 readings
SAMPLE_PERIOD_SEC = 1       # one reading per second
temp_buffer = deque(maxlen=N)
current_buffer = deque(maxlen=N)


def update_and_compute_slope(temp: float, current: float) -> Tuple[float, float]:
    """
    Uses the last N readings.
    Returns slopes scaled to "per 5 seconds":
    - thermal_slope_c_per_5s
    - current_slope_a_per_5s
    """
    temp_buffer.append(temp)
    current_buffer.append(current)

    if len(temp_buffer) < N:
        return 0.0, 0.0

    elapsed_seconds = (len(temp_buffer) - 1) * SAMPLE_PERIOD_SEC
    thermal_delta = temp_buffer[-1] - temp_buffer[0]
    current_delta = current_buffer[-1] - current_buffer[0]

    thermal_slope_per_5s = (thermal_delta / elapsed_seconds) * 5.0
    current_slope_per_5s = (current_delta / elapsed_seconds) * 5.0
    return thermal_slope_per_5s, current_slope_per_5s

In [20]:
FEATURE_COLS: List[str] = [
    "ambient_temp_c",
    "temperature_c",
    "temperature_rise_c",
    "current_a",
    "thermal_slope_c_per_5s",
    "current_slope_a_per_5s",
]


@dataclass
class SensorReading:
    ambient_temp_c: float
    temperature_c: float
    temperature_rise_c: float
    current_a: float
    thermal_slope_c_per_5s: float
    current_slope_a_per_5s: float

In [21]:
def generate_synthetic_data(n_samples: int = 20000, random_state: int = 42) -> pd.DataFrame:
    """
    Constraints requested:
    - Current: 0–40 A (<=20A tends to normal, >20A tends to overload)
    - Temperature: 40–100 °C
    - Temperature rise: normal around 50°C, >50°C tends abnormal

    Slopes are generated as "per 5 seconds" to match streaming scaling.
    """
    rng = np.random.default_rng(random_state)

    ambient_temp = np.clip(rng.normal(35.0, 4.5, n_samples), 20.0, 50.0)
    current = np.clip(rng.normal(20.0, 6.5, n_samples), 0.0, 40.0)

    prev_current = np.clip(current - rng.normal(0.0, 3.0, n_samples), 0.0, 40.0)
    current_slope_per_5s = (current - prev_current) * 5.0  # per-5s

    temperature_rise = np.clip(
        32.0 + 0.85 * current + rng.normal(0, 6.0, n_samples),
        20.0,
        65.0,
    )
    temperature = np.clip(ambient_temp + temperature_rise, 40.0, 100.0)

    prev_temp_rise = np.clip(
        32.0 + 0.85 * prev_current + rng.normal(0, 6.0, n_samples),
        20.0,
        65.0,
    )
    prev_temperature = np.clip(ambient_temp + prev_temp_rise, 40.0, 100.0)

    thermal_slope_per_5s = (temperature - prev_temperature) * 5.0  # per-5s

    hotspot_score = (
        0.10 * (temperature - 72)
        + 0.18 * (temperature_rise - 45)
        + 0.20 * thermal_slope_per_5s
        + 0.03 * current_slope_per_5s
        + rng.normal(0, 0.9, n_samples)
    )

    overload_score = (
        0.45 * (current - 18)
        + 0.15 * current_slope_per_5s
        + 0.04 * (temperature_rise - 45)
        + 0.02 * thermal_slope_per_5s
        + rng.normal(0, 0.6, n_samples)
    )

    hotspot_prob = 1 / (1 + np.exp(-hotspot_score))
    overload_prob = 1 / (1 + np.exp(-overload_score))

    hotspot = (rng.random(n_samples) < hotspot_prob).astype(int)
    overload = (rng.random(n_samples) < overload_prob).astype(int)

    return pd.DataFrame(
        {
            "ambient_temp_c": ambient_temp,
            "temperature_c": temperature,
            "temperature_rise_c": temperature_rise,
            "current_a": current,
            "thermal_slope_c_per_5s": thermal_slope_per_5s,
            "current_slope_a_per_5s": current_slope_per_5s,
            "hotspot": hotspot,
            "overload": overload,
        }
    )

In [22]:
df = generate_synthetic_data(n_samples=20000, random_state=7)

df.head()

,ambient_temp_c,temperature_c,temperature_rise_c,current_a,thermal_slope_c_per_5s,current_slope_a_per_5s,hotspot,overload
0,35.005536,97.070842,62.065307,26.498972,-14.645789,-0.624724,1,1
1,36.344355,75.121866,38.777511,16.507659,-43.395089,4.636734,0,0
2,33.766380,78.692786,44.926406,18.534972,-45.081282,-12.991783,0,1
3,30.992337,88.626657,57.634320,20.752557,29.720652,-22.566021,1,0
4,32.953981,90.074312,57.120331,20.534882,21.193593,2.752887,1,1


In [23]:
def train_models(df: pd.DataFrame, random_state: int = 7):
    X = df[FEATURE_COLS]
    y_hot = df["hotspot"].to_numpy()
    y_ovr = df["overload"].to_numpy()

    X_train, X_test, y_hot_train, y_hot_test, y_ovr_train, y_ovr_test = train_test_split(
        X, y_hot, y_ovr, test_size=0.25, random_state=random_state
    )

    param_grid = {
        "n_estimators": [600, 700, 800],
        "max_depth": [8, 12, None],
        "min_samples_leaf": [2, 4, 8],
        "max_features": ["sqrt", 0.8],
    }

    hotspot_grid = GridSearchCV(
        RandomForestClassifier(
            random_state=random_state,
            n_jobs=-1,
            class_weight="balanced",
        ),
        param_grid=param_grid,
        scoring="f1_macro",
        cv=5,
        n_jobs=-1,
        verbose=1,
    )
    hotspot_grid.fit(X_train, y_hot_train)

    overload_grid = GridSearchCV(
        RandomForestClassifier(
            random_state=random_state,
            n_jobs=-1,
            class_weight="balanced",
        ),
        param_grid=param_grid,
        scoring="f1_macro",
        cv=5,
        n_jobs=-1,
        verbose=1,
    )
    overload_grid.fit(X_train, y_ovr_train)

    return hotspot_grid.best_estimator_, overload_grid.best_estimator_, X_test, y_hot_test, y_ovr_test


hotspot_model, overload_model, X_test, y_hot_test, y_ovr_test = train_models(df, random_state=7)

hotspot_model, overload_model

Fitting 5 folds for each of 54 candidates, totalling 270 fits
Fitting 5 folds for each of 54 candidates, totalling 270 fits


(RandomForestClassifier(class_weight='balanced', max_depth=12,
                        min_samples_leaf=2, n_estimators=600, n_jobs=-1,
                        random_state=7),
 RandomForestClassifier(class_weight='balanced', max_depth=8, max_features=0.8,
                        min_samples_leaf=2, n_estimators=600, n_jobs=-1,
                        random_state=7))

In [24]:
hot_pred = hotspot_model.predict(X_test)
ovr_pred = overload_model.predict(X_test)

print("=== Hotspot Report ===")
print(classification_report(y_hot_test, hot_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_hot_test, hot_pred))

print("\n=== Overload Report ===")
print(classification_report(y_ovr_test, ovr_pred, digits=3))
print("Confusion Matrix:\n", confusion_matrix(y_ovr_test, ovr_pred))

=== Hotspot Report ===
              precision    recall  f1-score   support

           0      0.922     0.932     0.927      2160
           1      0.948     0.940     0.944      2840

    accuracy                          0.937      5000
   macro avg      0.935     0.936     0.935      5000
weighted avg      0.937     0.937     0.937      5000

Confusion Matrix:
 [[2013  147]
 [ 170 2670]]

=== Overload Report ===
              precision    recall  f1-score   support

           0      0.821     0.863     0.841      2013
           1      0.905     0.873     0.888      2987

    accuracy                          0.869      5000
   macro avg      0.863     0.868     0.865      5000
weighted avg      0.871     0.869     0.869      5000

Confusion Matrix:
 [[1738  275]
 [ 380 2607]]


In [ ]:
import os

os.makedirs("ml", exist_ok=True)

joblib.dump(hotspot_model, "ml/hotspot_model.pkl")
joblib.dump(overload_model, "ml/overload_model.pkl")

print("Models saved successfully")

Models saved successfully
